**Transform Orders Data-Explode Arrays**

In [0]:
dfOrdersData = spark.table("gizmobox_gr.silver.py_orders_json")
display(dfOrdersData)

**Access Elements From JSON Object**

In [0]:
from pyspark.sql import functions as f
dfOrders_normalize=(dfOrdersData
                    .select(
                        dfOrdersData.jsonObject.customer_id.alias("customer_id"),
                        dfOrdersData.jsonObject.transaction_timestamp.alias("transaction_timestamp"),
                        dfOrdersData.jsonObject.order_id.alias("order_id"),
                        dfOrdersData.jsonObject.order_date.alias("order_date"),
                        dfOrdersData.jsonObject.order_status.alias("order_status"),
                        dfOrdersData.jsonObject.payment_method.alias("payment_method"),
                        dfOrdersData.jsonObject.total_amount.alias("total_amount"),
                        dfOrdersData.jsonObject.items.alias("items")
                    ))
display(dfOrders_normalize)

**Deduplicate Array Elements**

In [0]:
dfOrders_deDupeArray=(dfOrders_normalize
                    .select(
                        dfOrders_normalize.customer_id,
                        dfOrders_normalize.transaction_timestamp,
                        dfOrders_normalize.order_id,
                        dfOrders_normalize.order_date,
                        dfOrders_normalize.order_status,
                        dfOrders_normalize.payment_method,
                        dfOrders_normalize.total_amount,
                        f.array_distinct(dfOrders_normalize.items).alias("items")
                    ))
display(dfOrders_deDupeArray)

**Explode Array**

In [0]:
dfOrders_explode=(dfOrders_deDupeArray
                    .select(
                        dfOrders_deDupeArray.customer_id,
                        dfOrders_deDupeArray.transaction_timestamp,
                        dfOrders_deDupeArray.order_id,
                        dfOrders_deDupeArray.order_date,
                        dfOrders_deDupeArray.order_status,
                        dfOrders_deDupeArray.payment_method,
                        dfOrders_deDupeArray.total_amount,
                        f.explode(dfOrders_deDupeArray.items).alias("item"),
                        f.explode(dfOrders_deDupeArray.items.details).alias("item_details")
                    ))
display(dfOrders_explode)

**Write Transformed Data To Silver Schema**

In [0]:
from pyspark.sql import functions as f
dfOrders_silver=(dfOrders_explode
                    .select(
                        dfOrders_explode.customer_id,
                        dfOrders_explode.transaction_timestamp,
                        dfOrders_explode.order_id,
                        dfOrders_explode.order_date,
                        dfOrders_explode.order_status,
                        dfOrders_explode.payment_method,
                        dfOrders_explode.total_amount,
                        dfOrders_explode.item.category.alias("item_category"),
                        dfOrders_explode.item_details.brand.alias("brand"),
                        dfOrders_explode.item_details.color.alias("color"),
                        dfOrders_explode.item.item_id.alias("item_id"),                        
                        f.col("item.name").alias("item_name"),
                        dfOrders_explode.item.price.alias("item_price"),
                        dfOrders_explode.item.quantity.alias("item_quantity")
                        ))

dfOrders_silver.writeTo("gizmobox_gr.silver.py_orders").createOrReplace()
display(spark.table("gizmobox_gr.silver.py_orders"))